# Análise de Dados - Salários em Data Science (2020 - 2024)

## Dataset: [Link](./data/data_science_salaries.csv)


## Importando Bibliotecas & Helpers


In [ ]:
from utils.helpers import (
    generate_unique_values_table,
    get_dimensions,
    COLUNAS_ORDENADAS,
)
from IPython.display import display, Markdown

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_squared_error,
    r2_score,
)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_style(style="whitegrid")


def set_styled_caption(styler, caption_text):
    """Aplica uma legenda e um estilo de preenchimento a um Styler do Pandas."""

    caption_style = [
        {
            "selector": "caption",
            "props": [
                ("padding-bottom", "8px"),
            ],
        }
    ]

    return styler.set_caption(f"<strong>{caption_text}</strong>").set_table_styles(
        caption_style
    )


usd_graph_formatter = ticker.FuncFormatter(lambda x, pos: f"${x:,.0f}")

## Lendo o dataset


In [ ]:
df_salary = pd.read_csv("./data/data_science_salaries.csv", sep=",")

In [ ]:
periodo = f"{df_salary['work_year'].min()} - {df_salary['work_year'].max()}"

markdown_text = f"""
**Dimensões:** {get_dimensions(df_salary)}  
**Período:** {periodo}  
"""

display(Markdown(markdown_text))

display(df_salary.head())

## Formatando os dados


In [ ]:
df_salary_formatted = df_salary.copy()

df_salary_formatted["work_year"] = df_salary_formatted["work_year"].astype("uint16")
df_salary_formatted["salary_in_usd"] = df_salary_formatted["salary_in_usd"].astype(
    "uint32"
)

df_salary_formatted["remote_ratio"] = df_salary_formatted["work_models"].map(
    {"On-site": 0, "Hybrid": 50, "Remote": 100}
)
df_salary_formatted["remote_ratio"] = df_salary_formatted["remote_ratio"].astype(
    "uint8"
)

# df_salary_formatted = df_salary_formatted.drop(columns=["work_models"])

# experience_level_map = {
#     "Entry-level": "EN",
#     "Mid-level": "MI",
#     "Senior-level": "SE",
#     "Executive-level": "EX",
# }
# employment_type_map = {
#     "Full-time": "FT",
#     "Part-time": "PT",
#     "Contract": "CT",
#     "Freelance": "FL",
# }
# company_size_map = {
#     "Small": "S",
#     "Medium": "M",
#     "Large": "L",
# }

# df_salary_formatted["experience_level"] = df_salary_formatted["experience_level"].map(
#     experience_level_map
# )
# df_salary_formatted["employment_type"] = df_salary_formatted["employment_type"].map(
#     employment_type_map
# )
# df_salary_formatted["company_size"] = df_salary_formatted["company_size"].map(
#     company_size_map
# )


numerical_cols = df_salary_formatted.select_dtypes(
    include=["int16", "int8", "int64", "float64", "uint8", "uint16", "uint32"]
).columns
categorical_cols = df_salary_formatted.select_dtypes(
    include=["category", "bool", "boolean", "object"]
).columns

for col in categorical_cols:
    df_salary_formatted[col] = df_salary_formatted[col].astype("category")

df_salary_formatted = df_salary_formatted[COLUNAS_ORDENADAS]
df_salary_formatted.info(memory_usage="deep")

old_memory = df_salary.memory_usage(deep=True).sum()
new_memory = df_salary_formatted.memory_usage(deep=True).sum()
reducao = (old_memory - new_memory) / old_memory * 100

memory_md = f"""
**Memória antes do processamento:** {old_memory / 1024:.2f} KB  
**Memória após o processamento:** {new_memory / 1024:.2f} KB  
**Redução:** {reducao:.2f}%  
"""

display(Markdown(memory_md))

df_salary_formatted.head()


### Valores únicos por categoria


In [ ]:
display(Markdown(generate_unique_values_table(df_salary_formatted, 200, show_count=True)))


### Colunas Numéricas vs Categóricas


In [ ]:
variable_type_summary = (
    """
| Tipo de Variável | Colunas |
|------------------|---------|
"""
    + f"| Numéricas        | {', '.join(numerical_cols)} |\n"
    f"| Categóricas      | {', '.join(categorical_cols)} |\n"
)

display(Markdown(variable_type_summary))

## **Parte 1 – Análise Descritiva e Exploratória (EDA)**

**Objetivo:** Identificar o comportamento geral dos salários e fatores relacionados ao perfil profissional.


### **Métricas Obrigatórias**

#### Média, mediana e desvio-padrão de salary_in_usd.  

In [ ]:
salary_stats_geral = df_salary_formatted["salary_in_usd"].agg(["mean", "median", "std"])

translation_map = {
    "mean": "Média",
    "median": "Mediana",
    "std": "Desvio Padrão",
}

salary_stats_geral.index = salary_stats_geral.index.map(translation_map)

display(
    set_styled_caption(
        salary_stats_geral.to_frame("Geral")
        .reset_index()
        .rename(columns={"index": "Métrica"})
        .style.format({"Geral": "${:,.2f}"})
        .set_properties(subset=["Métrica"], **{"text-align": "left"})
        .hide(axis="index"),
        "Estatísticas Gerais de Salário (USD)",
    )
)

#### Distribuição percentual de experience_level, employment_type e company_size.

In [ ]:
dist_experience_level = (
    df_salary_formatted["experience_level"].value_counts(normalize=True) * 100
).round(2)
dist_employment_type = (
    df_salary_formatted["employment_type"].value_counts(normalize=True) * 100
).round(2)
dist_company_size = (
    df_salary_formatted["company_size"].value_counts(normalize=True) * 100
).round(2)


def show_percentual_distribution(series, label):
    styled = (
        series.rename_axis(label)
        .to_frame("Proporção (%)")
        .reset_index()
        .style.format({"Proporção (%)": "{:.2f}%"})
        .set_properties(subset=[label], **{"text-align": "left"})
        .hide(axis="index")
    )
    display(
        set_styled_caption(
            styled,
            f"Distribuição por {label}:"
        )
    )


show_percentual_distribution(dist_experience_level, "Nível de Experiência")
show_percentual_distribution(dist_employment_type, "Tipo de Contrato")
show_percentual_distribution(dist_company_size, "Porte da Empresa")


#### Top 5 cargos (job_title) com maior média salarial.

In [ ]:
top5_job_titles = (
    df_salary_formatted.groupby("job_title", observed=True)["salary_in_usd"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

display(
    set_styled_caption(
        top5_job_titles.reset_index()
        .rename(columns={"salary_in_usd": "Média Salarial (USD)", "job_title": "Cargo"})
        .style.format({"Média Salarial (USD)": "${:,.2f}"})
        .set_properties(subset=["Cargo"], **{"text-align": "left"})
        .set_table_styles(
            [
                {
                    "selector": "th.col_heading.level0.col0",
                    "props": [("text-align", "left")],
                }
            ],
            overwrite=False,
        )
        .hide(axis="index"),
        "Top 5 cargos com maior média salarial:",
    )
)

#### Correlação entre remote_ratio, company_size e salary_in_usd.

In [ ]:
corr_cols = ["remote_ratio", "company_size", "salary_in_usd"]

company_size_map = {"Small": 0, "Medium": 1, "Large": 2}
df_corr = df_salary_formatted.copy()
df_corr["company_size_num"] = df_corr["company_size"].map(company_size_map)

corr_matrix = df_corr[["remote_ratio", "company_size_num", "salary_in_usd"]].corr()

display(
    set_styled_caption(
        corr_matrix.style.format("{:.3f}").background_gradient(
            cmap="RdBu_r", axis=None
        ),
        "Matriz de correlação entre remote_ratio, company_size e salary_in_usd:",
    )
)


plt.figure(figsize=(6, 5))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="RdBu_r",
    center=0,
    square=True,
    fmt=".3f",
    cbar_kws={"shrink": 0.8},
)
plt.title(
    "Matriz de Correlação - Variáveis Numéricas",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
plt.tight_layout()
plt.savefig("./imgs/correlation_matrix-company_size-remote_ratio-salary.png", dpi=300)
plt.show()

#### Boxplot de salary_in_usd por experience_level e company_location.

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df_salary_formatted,
    x="experience_level",
    y="salary_in_usd",
)
plt.title(
    "Boxplot de Salário (USD) por Nível de Experiência", fontsize=14, fontweight="bold"
)
plt.xlabel("Nível de Experiência")
plt.ylabel("Salário em USD")

ax = plt.gca()
ax.yaxis.set_major_formatter(usd_graph_formatter)

plt.tight_layout()
plt.show()


top5_locations = df_salary_formatted["company_location"].value_counts().head(5).index

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df_salary_formatted[
        df_salary_formatted["company_location"].isin(top5_locations)
    ],
    x="company_location",
    y="salary_in_usd",
    order=top5_locations,
)
plt.title(
    "Boxplot de Salário (USD) por Localização da Empresa (Top 5 Países com mais Registros)",
    fontsize=14,
    fontweight="bold",
)

plt.xlabel("Localização da Empresa")
plt.ylabel("Salário em USD")
ax = plt.gca()
ax.yaxis.set_major_formatter(usd_graph_formatter)
plt.tight_layout()
plt.show()


def plot_boxplot_experience_by_country(df, country):
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df[df["company_location"] == country],
        x="experience_level",
        y="salary_in_usd",
    )
    plt.title(
        f"Boxplot de Salário (USD) por Nível de Experiência - {country}",
        fontsize=14,
        fontweight="bold",
    )
    ax = plt.gca()
    formatter = ticker.FuncFormatter(lambda x, pos: f"${x:,.0f}")
    ax.yaxis.set_major_formatter(formatter)
    plt.xlabel("Nível de Experiência")
    plt.ylabel("Salário em USD")
    plt.tight_layout()
    plt.show()


plot_boxplot_experience_by_country(df_salary_formatted, "Brazil")

### **Perguntas executivas**


#### **Q1**: Qual é a média e o desvio-padrão do $salary\_in\_usd$ para cada categoria de $experience\_level$? Qual nível apresenta maior variabilidade salarial e o que isso indica sobre o mercado?


In [ ]:
salary_stats_exp = (
    df_salary_formatted.groupby("experience_level", observed=True)["salary_in_usd"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

maior_variabilidade = salary_stats_exp["std"].idxmax()
variabilidade_valor = salary_stats_exp.loc[maior_variabilidade, "std"]

dist_percentual = (
    df_salary_formatted["experience_level"].value_counts(normalize=True) * 100
).round(2)

display(
    set_styled_caption(
        salary_stats_exp.reset_index()
        .rename(
            columns={
                "experience_level": "Nível de Experiência",
                "mean": "Média",
                "median": "Mediana",
                "std": "Desvio Padrão",
                "count": "Contagem",
            }
        )
        .style.format(
            {
                "Média": "${:,.2f}",
                "Mediana": "${:,.2f}",
                "Desvio Padrão": "${:,.2f}",
                "Contagem": "{:d}",
            }
        )
        .set_properties(subset=["Nível de Experiência"], **{"text-align": "left"})
        .hide(axis="index"),
        "Estatísticas por nível de experiência:",
    )
)

display(
    Markdown(
        f"**Maior variabilidade salarial:** {maior_variabilidade} (desvio-padrão = {variabilidade_valor:,.2f})"
    )
)

#### **Resposta**: 
O Nível Intermediário (Mid-level) é o que apresenta a maior variabilidade salarial (desvio-padrão de $71.783,36), superando por pouco o Nível Executivo.

Isso indica que o mercado para "Mid-level" é o menos padronizado. O termo abrange uma gama muito ampla de funções e especialidades. Na prática, há uma enorme disparidade entre os salários mais baixos e os mais altos dentro dessa mesma categoria.

#### **Q2**: Qual tipo de contrato ($employment\_type$) apresenta maior média salarial? Essa diferença se mantém entre portes de empresa ($company\_size$)?


In [ ]:
mean_salary_by_contract = (
    df_salary_formatted.groupby("employment_type", observed=True)["salary_in_usd"]
    .mean()
    .sort_values(ascending=False)
)

display(
    set_styled_caption(
        mean_salary_by_contract.reset_index()
        .rename(
            columns={
                "employment_type": "Tipo de Contrato",
                "salary_in_usd": "Média Salarial (USD)",
            }
        )
        .style.format({"Média Salarial (USD)": "${:,.2f}"})
        .hide(axis="index"),
        "Média salarial por tipo de contrato:",
    )
)

mean_salary_by_contract_and_size = (
    df_salary_formatted.groupby(["employment_type", "company_size"], observed=True)[
        "salary_in_usd"
    ]
    .mean()
    .unstack("company_size")
    .loc[mean_salary_by_contract.index]
)

display(
    set_styled_caption(
        mean_salary_by_contract_and_size.rename_axis("Tipo de Contrato")
        .rename(columns={"Small": "Pequena", "Medium": "Média", "Large": "Grande"})
        .rename_axis(columns=None)
        .style.format("${:,.2f}"),
        "Média salarial por tipo de contrato e porte da empresa",
    )
)

#### **Resposta**:

O **Full-time** (período integral) é o tipo de contrato com a maior média salarial geral, com **$146.035,00**.

No entanto, essa liderança **não se mantém** em todos os portes de empresa.

* Em empresas **Grandes** e **Médias**, o "Full-time" de fato lidera.
* Em empresas **Pequenas**, o "Contract" (contrato por projeto/temporário) apresenta uma média salarial significativamente maior ($128.528,57) do que o "Full-time" ($89.037,35).

#### **Q3**: Compare a média de $salary\_in\_usd$ por $company\_location$. Quais países se destacam por maiores ou menores salários? Que fatores econômicos podem justificar essa diferença?


In [ ]:
mean_salary_by_location = (
    df_salary_formatted.groupby("company_location", observed=True)["salary_in_usd"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

display(
    set_styled_caption(
        mean_salary_by_location.reset_index()
        .rename(
            columns={
                "company_location": "Localização da Empresa",
                "mean": "Média Salarial (USD)",
                "count": "Nº de Registros",
            }
        )
        .style.format({"Média Salarial (USD)": "${:,.2f}", "Nº de Registros": "{:d}"})
        .hide(axis="index"),
        "Média salarial e quantidade de registros por localização da empresa:",
    )
)

plt.figure(figsize=(14, 6))
top_n = 10
sns.barplot(
    data=mean_salary_by_location.head(top_n).reset_index(),
    x="company_location",
    y="mean",
    order=mean_salary_by_location.head(top_n).index,
)
plt.title(
    f"Top {top_n} Países com Maior Média Salarial (USD)", fontsize=14, fontweight="bold"
)
plt.xlabel("Localização da Empresa")
plt.ylabel("Média Salarial (USD)")
ax = plt.gca()
ax.yaxis.set_major_formatter(usd_graph_formatter)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
bottom_n = 10
sns.barplot(
    data=mean_salary_by_location.tail(bottom_n).reset_index(),
    x="company_location",
    y="mean",
    order=mean_salary_by_location.tail(bottom_n).index,
)
plt.title(
    f"Top {bottom_n} Países com Menor Média Salarial (USD)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Localização da Empresa")
plt.ylabel("Média Salarial (USD)")
ax = plt.gca()
ax.yaxis.set_major_formatter(usd_graph_formatter)
plt.tight_layout()
plt.show()


#### **Resposta**:

**Maiores Salários**: Qatar ($300.000), Israel ($217.332) e Estados Unidos ($157.073)  
(Ps: Porto Rico também se destaca ($167,500), mas é um território dos EUA).

**Menores Salários**: Equador ($16.000), Moldova ($18.000) e Honduras ($20.000)

#### **Q4**: Analise a correlação entre $remote\_ratio$ e $salary\_in\_usd$. Há indícios de que o trabalho remoto impacte positivamente ou negativamente o salário? Explique.


In [ ]:
sns.stripplot(
    data=df_salary_formatted,
    x="work_models",
    y="salary_in_usd",
    alpha=0.3,
    order=["On-site", "Hybrid", "Remote"],
    jitter=0.1,
)
plt.title(
    "Relação entre o Salário e o Nível de Trabalho Remoto",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Modelo de Trabalho")
plt.ylabel("Salário em USD")

ax = plt.gca()
ax.yaxis.set_major_formatter(usd_graph_formatter)

plt.tight_layout()
plt.show()


correlacao_remote_salary = df_salary_formatted["remote_ratio"].corr(
    df_salary_formatted["salary_in_usd"]
)

display(
    Markdown(
        f"**Correlação entre Nível Remoto e Salário** {correlacao_remote_salary:.3f} <br>"
        "Valores próximos de 0 indicam pouca ou nenhuma relação linear entre o nível de trabalho remoto e o salário."
    )
)


presencial_salaries = df_salary_formatted[
    df_salary_formatted["work_models"] == "On-site"
]["salary_in_usd"]
hibrido_salaries = df_salary_formatted[df_salary_formatted["work_models"] == "Hybrid"][
    "salary_in_usd"
]
remoto_salaries = df_salary_formatted[df_salary_formatted["work_models"] == "Remote"][
    "salary_in_usd"
]

kruskal_stat, kruskal_p = stats.kruskal(
    presencial_salaries, hibrido_salaries, remoto_salaries
)

alpha = 0.05
display(
    Markdown(
        f"**Teste de Kruskal-Wallis**  \n"
        f"Estatística H: {kruskal_stat:.4f}  \n"
        f"Valor-p: {kruskal_p:.4e}  \n"
        f"Nível de significância: {alpha:.2f}  \n"
        + (
            "\n**Conclusão:** Rejeitamos a hipótese nula.  \n"
            "Há diferença significativa entre pelo menos dois grupos."
            if kruskal_p < alpha
            else "\n**Conclusão:** Falhamos em rejeitar a hipótese nula.  \nNão há evidência de diferença significativa entre os grupos."
        )
    )
)



##### **Resposta**: Considerando a correlação de -0.088, embora o sinal seja tecnicamente negativo, a força dessa correlação é tão baixa que, para todos os efeitos práticos, esse dado não mostra uma relação relevante entre o nível de trabalho remoto e o salário. Ou seja, não há um indício forte de que um impacte o outro.

#### **Q5**: Quais cargos ($job\_title$) aparecem entre os cinco mais bem remunerados? Eles correspondem a posições consolidadas ou emergentes?


In [ ]:
top5_job_titles_with_count = (
    df_salary_formatted.groupby("job_title", observed=True)
    .agg({"salary_in_usd": "mean", "job_title": "count"})
    .rename(columns={"salary_in_usd": "Média Salarial (USD)", "job_title": "Contagem"})
    .sort_values("Média Salarial (USD)", ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={"job_title": "Cargo"})
)

display(
    set_styled_caption(
        top5_job_titles_with_count
        .style.format({"Média Salarial (USD)": "${:,.2f}", "Contagem": "{:d}"})
        .set_properties(subset=["Cargo"], **{"text-align": "left"})
        .hide(axis="index"),
        "Top 5 cargos mais bem remunerados (com contagem):",
    )
)

##### **Resposta**: 
Os cargos listados entre os cinco mais bem remunerados são:

- **Analytics Engineering Manager**: $399,880.00  
- **Data Science Tech Lead**: $375,000.00  
- **Managing Director Data Science**: $280,000.00  
- **AWS Data Architect**: $258,000.00  
- **Cloud Data Architect**: $250,000.00  

Essas funções correspondem principalmente a posições consolidadas e de alta senioridade, como diretoria, liderança técnica e arquitetura de dados. Algumas delas, como AWS Data Architect e Cloud Data Architect, também refletem posições emergentes relacionadas à computação em nuvem e à modernização de infraestrutura de dados.

## **Parte 2 – Modelagem Preditiva (Regressão Linear e Logística)** 

In [ ]:
from IPython.display import display, Markdown

print(df_salary_formatted.columns.tolist())

categorical_cols = [
    "job_title",
    "experience_level",
    "employment_type",
    "work_models",
    "employee_residence",
    "company_location",
    "company_size",
]

# Criar o novo DataFrame com as dummies
df_dummies = pd.get_dummies(df_salary_formatted, columns=categorical_cols)
feature_names = df_dummies.columns.tolist()
feature_names.remove("salary_in_usd")
feature_names.remove("remote_ratio")

X = df_dummies[feature_names]
y = df_dummies["salary_in_usd"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


lr = LinearRegression()
lr.fit(X_train_scaled, y_train)


y_pred = lr.predict(X_test_scaled)

mae = np.mean(np.abs(y_test - y_pred))
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

display(
    Markdown(
        f"**Modelo de Regressão Linear - Desempenho no Conjunto de Teste**  \n"
        f"- MAE: {mae:,.2f}  \n"
        f"- MSE: {mse:,.2f}  \n"
        f"- RMSE: {rmse:,.2f}  \n"
        f"- R²: {r2:.3f}  \n"
    )
)


top_n = 10

coefs = lr.coef_
nome = "Regressão Linear"

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs, "abs_coef": np.abs(coefs)})
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)

coef_df["count"] = coef_df["feature"].apply(
    lambda x: X_train[x].sum() if x in X_train.columns else 0
)


style_format = {"coef": "{:.2f}", "count": "{:.0f}"}


top_positive = coef_df.sort_values("coef", ascending=False).head(top_n)

display(Markdown(f"### 📈 {nome} - Top {top_n} Coeficientes (Impacto Positivo)"))
display(
    top_positive[["feature", "coef", "count"]].style.format(
        style_format  ## type: ignore
    )
)


top_negative = coef_df.sort_values("coef", ascending=True).head(top_n)

display(Markdown(f"### 📉 {nome} - Top {top_n} Coeficientes (Impacto Negativo)"))
display(
    top_negative[["feature", "coef", "count"]].style.format(
        style_format  ## type: ignore
    )
)